# Part 4 · Explore saved results
Reopen results without rerunning the simulation. This works with a run folder or results ZIP. Checksums are verified first.

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util

# In Colab, the public GitHub version must already contain this release.
INSTALL_PACKAGE = True
REPO_URL = "https://github.com/hylixinsights/VirtualTissue-scRNAseq.git"
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((d for d in candidates if (d / "pyproject.toml").exists() and (d / "src/vtsc").exists()), None)
if ROOT is None:
    ROOT = Path("/content/VirtualTissue-scRNAseq") if IN_COLAB else Path.home() / "VirtualTissue-scRNAseq"
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
if INSTALL_PACKAGE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT) + "[notebooks]"], check=True)
sys.path.insert(0, str(ROOT / "src"))
print("Code:", ROOT)
# Leave WORK below local, or explicitly mount Drive and choose a Drive path.
WORK = Path(os.environ.get("VTSC_NOTEBOOK_WORK", str(ROOT / "outputs/notebook_demo")))
WORK.mkdir(parents=True, exist_ok=True)
print("Files:", WORK)


In [ ]:
RESULTS = WORK / 'baseline'  # or Path('/path/to/run.zip')
EXPLORER_COPY = WORK / 'reopened_explorer.html'


In [ ]:
from vtsc.results import read_run, write_html
import pandas as pd
result = read_run(RESULTS)
print('Verified:', result['summary'])
print('Assumptions:', result['model']['rate_provenance'])
pd.DataFrame(result['trajectory']).head()


In [ ]:
from IPython.display import HTML, display
write_html(result, EXPLORER_COPY)
display(HTML(EXPLORER_COPY.read_text()))
print('For reliable full-screen interaction, download/open this file in your browser:', EXPLORER_COPY)


In [ ]:
# Full event history; change CELL_ID to a measured or simulated ID.
CELL_ID = result['initial_cells'][0]['cell_id']
events = pd.DataFrame(result['events'], columns=['event_id','time_hours','event','cell_id','parent_id','from_state','to_state'])
events.loc[(events['cell_id'] == CELL_ID) | (events['parent_id'] == CELL_ID)]


The live-agent grid is an index layout, not geometry inferred from scRNA-seq. Counts and events are simulated under user assumptions. Shared files retain source terms and identifiers; check permissions before sharing them.